In [94]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os
from langgraph.checkpoint.memory import InMemorySaver

In [95]:
load_dotenv()


True

In [96]:
google_model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.9
)

openai_model = ChatOpenAI(
    model="gpt-5.4-mini",
    temperature=0.9
)

In [97]:
from google import genai

In [98]:
client = genai.Client()

In [99]:
interaction =client.interactions.create(
    model="gemini-3.5-flash-lite",
    input="Explain the demand for rust programming language in near future compared to other prominent language"
)

In [100]:
class JokeState(TypedDict):

    topic: str
    joke_open: str
    joke_google: str
    exp_open: str
    exp_google: str


    

In [101]:
def generate_joke(state: JokeState):

    prompt = f'Generate a joke on the topic {state["topic"]}'
    response_google = google_model.invoke(prompt).content
    response_open = openai_model.invoke(prompt).content

    return {'joke_open': response_open, "joke_google": response_google}

In [102]:
def generate_explanation(state: JokeState):
    prompt_open = f'write an explanation for the joke - {state["joke_open"]}'
    prompt_google = f'write an explanation for the joke - {state["joke_google"]}'
    response_google = google_model.invoke(prompt_google).content
    response_open = openai_model.invoke(prompt_open).content

    return {'exp_open': response_open, 'exp_google': response_google}

In [103]:
graph = StateGraph(JokeState)

In [104]:
graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

In [105]:
workflow = graph.compile(checkpointer=checkpointer)

In [106]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'Football'}, config=config1)

/home/j/miniconda3/envs/langgraph-test/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/home/j/miniconda3/envs/langgraph-test/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{'topic': 'Football',
 'joke_open': 'Why did the football team go to the bank?  \nBecause they wanted their quarterback!',
 'joke_google': [{'type': 'text',
   'text': "Why do football players do so well on their taxes?\n\nBecause they're great at handling drafts!",
   'extras': {'signature': 'EmAKXgFpFH0Tb0wvt//wYE7+csGHabP0NbGveliikHFh19LhemUBCla+FbCm+QMeKwFGIOFn1m5v2O74g/+auMEhnJrByWi7ip+aRR4M0yLImLamWZ69aAEWd6XUTTAGpbo='}}],
 'exp_open': 'Here’s the joke explained:\n\n- In football, a **quarterback** is a key player who leads the offense.\n- In a **bank**, a **quarterback** sounds like someone who would be needed in a financial sense — like a “quarter” being money.\n\nThe joke is a **pun**:\n- “Quarterback” sounds like it could be understood as “quarter back.”\n- So the punchline makes it seem like the football team went to the bank to get a “quarter” back, when really it’s referring to the football position.\n\nIt’s funny because it mixes up **football language** with **bank/money

In [107]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Football', 'joke_open': 'Why did the football team go to the bank?  \nBecause they wanted their quarterback!', 'joke_google': [{'type': 'text', 'text': "Why do football players do so well on their taxes?\n\nBecause they're great at handling drafts!", 'extras': {'signature': 'EmAKXgFpFH0Tb0wvt//wYE7+csGHabP0NbGveliikHFh19LhemUBCla+FbCm+QMeKwFGIOFn1m5v2O74g/+auMEhnJrByWi7ip+aRR4M0yLImLamWZ69aAEWd6XUTTAGpbo='}}], 'exp_open': 'Here’s the joke explained:\n\n- In football, a **quarterback** is a key player who leads the offense.\n- In a **bank**, a **quarterback** sounds like someone who would be needed in a financial sense — like a “quarter” being money.\n\nThe joke is a **pun**:\n- “Quarterback” sounds like it could be understood as “quarter back.”\n- So the punchline makes it seem like the football team went to the bank to get a “quarter” back, when really it’s referring to the football position.\n\nIt’s funny because it mixes up **football language** with 

In [108]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Football', 'joke_open': 'Why did the football team go to the bank?  \nBecause they wanted their quarterback!', 'joke_google': [{'type': 'text', 'text': "Why do football players do so well on their taxes?\n\nBecause they're great at handling drafts!", 'extras': {'signature': 'EmAKXgFpFH0Tb0wvt//wYE7+csGHabP0NbGveliikHFh19LhemUBCla+FbCm+QMeKwFGIOFn1m5v2O74g/+auMEhnJrByWi7ip+aRR4M0yLImLamWZ69aAEWd6XUTTAGpbo='}}], 'exp_open': 'Here’s the joke explained:\n\n- In football, a **quarterback** is a key player who leads the offense.\n- In a **bank**, a **quarterback** sounds like someone who would be needed in a financial sense — like a “quarter” being money.\n\nThe joke is a **pun**:\n- “Quarterback” sounds like it could be understood as “quarter back.”\n- So the punchline makes it seem like the football team went to the bank to get a “quarter” back, when really it’s referring to the football position.\n\nIt’s funny because it mixes up **football language** with

In [109]:


config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'Cricket'}, config=config2)



/home/j/miniconda3/envs/langgraph-test/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/home/j/miniconda3/envs/langgraph-test/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{'topic': 'Cricket',
 'joke_open': 'Why did the cricket player bring string to the match?\n\nBecause he wanted to tie the score!',
 'joke_google': [{'type': 'text',
   'text': 'Why do cricket players love to drink tea?\n\nBecause they know how to handle a good *break*, even if it takes five days!',
   'extras': {'signature': 'EmAKXgFpFH0ThErneJ2pahPoM/mBeZNYRdiFFRwsFpdgCQsxDKUZUS/X1qEAp44+DajFOvzrc1FdQfvPR0z7u3ABXNDFtE1PZz0Wk4uNhFJEuCHNt0SixxQxxYt0leM6WP4='}}],
 'exp_open': 'Here’s the explanation:\n\nThe joke plays on the phrase **“tie the score,”** which means to make the game even by matching the other team’s points. But the word **“tie”** also means to **bind or fasten something with string**. So the punchline is funny because the cricket player brought string to literally “tie” something, while the real meaning is that he wanted to equal the score.\n\nIf you want, I can also make this explanation simpler or more kid-friendly.',
 'exp_google': [{'type': 'text',
   'text': 'This jok

In [110]:
workflow.get_state(config2)


StateSnapshot(values={'topic': 'Cricket', 'joke_open': 'Why did the cricket player bring string to the match?\n\nBecause he wanted to tie the score!', 'joke_google': [{'type': 'text', 'text': 'Why do cricket players love to drink tea?\n\nBecause they know how to handle a good *break*, even if it takes five days!', 'extras': {'signature': 'EmAKXgFpFH0ThErneJ2pahPoM/mBeZNYRdiFFRwsFpdgCQsxDKUZUS/X1qEAp44+DajFOvzrc1FdQfvPR0z7u3ABXNDFtE1PZz0Wk4uNhFJEuCHNt0SixxQxxYt0leM6WP4='}}], 'exp_open': 'Here’s the explanation:\n\nThe joke plays on the phrase **“tie the score,”** which means to make the game even by matching the other team’s points. But the word **“tie”** also means to **bind or fasten something with string**. So the punchline is funny because the cricket player brought string to literally “tie” something, while the real meaning is that he wanted to equal the score.\n\nIf you want, I can also make this explanation simpler or more kid-friendly.', 'exp_google': [{'type': 'text', 'text': '

In [111]:
workflow.get_state({"configurable": {"thread_id": "2", 'checkpoint_id': '1f1d9b27-6211-6299-8001-8d7253eac817'}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f1d9b27-6211-6299-8001-8d7253eac817'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [112]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Cricket', 'joke_open': 'Why did the cricket player bring string to the match?\n\nBecause he wanted to tie the score!', 'joke_google': [{'type': 'text', 'text': 'Why do cricket players love to drink tea?\n\nBecause they know how to handle a good *break*, even if it takes five days!', 'extras': {'signature': 'EmAKXgFpFH0ThErneJ2pahPoM/mBeZNYRdiFFRwsFpdgCQsxDKUZUS/X1qEAp44+DajFOvzrc1FdQfvPR0z7u3ABXNDFtE1PZz0Wk4uNhFJEuCHNt0SixxQxxYt0leM6WP4='}}], 'exp_open': 'Here’s the explanation:\n\nThe joke plays on the phrase **“tie the score,”** which means to make the game even by matching the other team’s points. But the word **“tie”** also means to **bind or fasten something with string**. So the punchline is funny because the cricket player brought string to literally “tie” something, while the real meaning is that he wanted to equal the score.\n\nIf you want, I can also make this explanation simpler or more kid-friendly.', 'exp_google': [{'type': 'text', 'text': 

In [113]:
workflow.update_state({"configurable": {"thread_id": "2", 'checkpoint_id': '1f1d9b27-6211-6299-8001-8d7253eac817', "checkpoint_ns": ""}}, {'topic':'Tennis'})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1da08e-7ac7-67b7-8000-f88278e8470f'}}

In [114]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Tennis'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1da08e-7ac7-67b7-8000-f88278e8470f'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-11-06T17:38:40.662917+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1d9b27-6211-6299-8001-8d7253eac817'}}, tasks=(PregelTask(id='144efdf5-27bc-36d6-deda-81ba327c5cc7', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'Cricket', 'joke_open': 'Why did the cricket player bring string to the match?\n\nBecause he wanted to tie the score!', 'joke_google': [{'type': 'text', 'text': 'Why do cricket players love to drink tea?\n\nBecause they know how to handle a good *break*, even if it takes five days!', 'extras': {'signature': 'EmAKXgFpFH0ThErneJ2pahPoM/mBeZNYRdiFFR

In [115]:
# Fault TOlerance
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [116]:
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [117]:
#define steps
def step_1(state: CrashState) ->CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

In [118]:
def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(30)
    return {"step2":"done"}


In [119]:
def step_3(state: CrashState) -> CrashState:
    print("Step 3 executed")
    return {"done": True}

In [120]:
#3 Build the graph

builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [121]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")


▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
Step 3 executed


In [122]:


# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)




🔁 Re-running the graph to demonstrate fault tolerance...

✅ Final State: {'input': 'start', 'step1': 'done', 'step2': 'done'}


In [123]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1da08f-99bc-6366-8003-b54bfdae7549'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-11-06T17:39:10.752434+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1da08f-99b9-6ddb-8002-ebd8ebb073f5'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1da08f-99b9-6ddb-8002-ebd8ebb073f5'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-11-06T17:39:10.751466+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1da08e-7b9e-61cc-8001-fb3f9bcec07a'}}, tasks=(PregelTask(id='e4fe95b3-f34e-9ed9-a6ca-f9d8c4e4ab0